## Structured Output

Modules can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain support multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C2BE71CE50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C2BF854F10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")
    

In [6]:
model_with_structured_output = model.with_structured_output(Movie)
response = model_with_structured_output.invoke("Provide details about the movie Inception.")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [4]:
model.invoke("Provide details about the movie Inception.")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know. Directed by Christopher Nolan, right? It\'s a sci-fi action film. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who enters people\'s dreams to steal secrets.\n\nThe plot involves a technique called "inception," which is the opposite of extraction. Instead of stealing secrets, you plant an idea. The team has to perform this heist on a target. The target is Robert Fischer, the heir to a huge corporation. The goal is to make him inherit the company and then have him believe he wanted to take over, not that it was implanted.\n\nThere\'s a concept of layers in the movie. Each level of the dream is deeper, and the time slows down. So the deeper they go, the longer it takes in real time. The team uses different tools like the PASIV device, which is a machine that allows them to share a dream. They also use totems, like Cobb\'s spinning top,

### Message Output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with its details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 1 to 10")

model_with_structured_output = model.with_structured_output(Movie, include_raw = True)

response = model_with_structured_output.invoke("Provide details about the movie Inception.")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. I need to use the Movie function to provide that information. Let me check what parameters are required: title, year, director, and rating. \n\nFirst, the title is obviously "Inception". The release year was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. I should make sure those are the correct details. Let me confirm: yes, Inception came out in 2010, directed by Christopher Nolan, and the rating is indeed 8.8. \n\nNow, I need to structure the function call correctly. The required fields are title, year, director, and rating. All of these are present in the information I gathered. I\'ll format the JSON accordingly, making sure the types are correct—year is an integer, rating is a number. \n\nI should avoid any typos. Let me double-check the spelling of the director\'s name and the movie title. Everything lo

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(..., description="List of main actors and their roles")
    gerne: str = Field(..., description="The genre of the movie")
    budget: float = Field(..., description="The budget of the movie in millions USD")

model_with_structured_output = model.with_structured_output(MovieDetails)

response = model_with_structured_output.invoke("Provide detailed information about the movie Inception, including its cast and budget.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Caius'), Actor(name='Ken Watanabe', role='Professor Fujita')], gerne='Science Fiction', budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with its details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie on a scale of 1 to 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [10]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    cast: Annotated[list[Actor], ..., "List of main actors and their roles"]
    gerne: Annotated[str, ..., "The genre of the movie"]
    budget: Annotated[float, ..., "The budget of the movie in millions USD"]

model_with_structured_output = model.with_structured_output(MovieDetails)

response = model_with_structured_output.invoke("Provide detailed information about the movie Inception, including its cast and budget.")
response

{'budget': 160,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'}],
 'gerne': 'Science Fiction',
 'title': 'Inception',
 'year': 2010}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [ ]:
# Using Pydantic models for structured output in an agent
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """A contact information entry."""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo   # Auto-selets ProviderStrategy
)

result = agent.invoke({
    message: [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

print(result["structured_output "])
# ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

In [ ]:
# Using TypedDict for structured output in an agent
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """A contact information entry."""
    name: Annotated[str, ..., "The name of the contact"]
    email: Annotated[str, ..., "The email address of the contact"]
    phone: Annotated[str, ..., "The phone number of the contact"]

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo  # Auto-selets ProviderStrategy
)

result = agent.invoke({
    message: [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

result["structured_output "]
# {'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}

In [ ]:
# DataClasses

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """A contact information entry."""
    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo  # Auto-selets ProviderStrategy
)

result = agent.invoke({
    message: [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

result["structured_output"]
# ContactInfo(name = 'John Doe', email= 'john.doe@example.com', phone = '123-456-7890')